# Run-book 145-X3 — `download_shard` on AML after the shard-row check

**What this proves:** gate 4 for #145 slice X3 (PR #180, spec 37 Amendment A1). One real MPC shard
job, running **this branch's** fsd on an AML node, still downloads and catalogs a granule correctly
now that `download_shard` checks every shard row for the 16 keys and fills in no defaults. The
refusal itself (a row with a missing key) is proved locally by test T3 (`run_shard`), so this
run-book covers the happy path only (A1, How to verify).

**Claude does not run this** (spec 24). You run it cell by cell and paste back the **Summary**
cell's output, or a failing cell's message. Every check is a plain `assert` you can read: the
text above a check cell says what PASS means, and the `assert` says the same thing in code.

**Needs:** your Azure login (`az login`) with access to the `rise` workspace and blob; `fsd`
config set (`fsd init`) including `image_registry`; network. **Time:** a new image build is
~10-20 min (estimate, from `docs/howto/build-the-images.md`) plus one AML job (~5-10 min incl.
node start; estimate). **Data:** one Sentinel-2 L2A granule, band B04 only (~120 MB, estimate).

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `fsd/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with fsd's dependencies.
- Open VS Code on the **`fsd/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `fsd/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `fsd/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `fsd` gets imported.** fsd is installed from the *main* checkout, so inside a
worktree a plain `import fsd` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which fsd. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be fsd's own virtual environment, fsd/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not fsd's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the fsd checkout that owns the venv

# 2. Which fsd? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "fsd").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import fsd

assert Path(fsd.__file__).is_relative_to(CHECKOUT / "src"), (
    f"fsd was already imported from {fsd.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("fsd    :", fsd.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else. **Replace `<your-user>`** in
`AZ_ROOT` with your own blob prefix.

In [ ]:
# Settings
import datetime
import json

import fsd
from fsd.storage.azure import configure_storage

cfg = fsd.config.load()          # subscription, resource group, workspace, cluster, uami, image_registry
assert cfg.image_registry, "set image_registry: `fsd init --set image_registry=abfss://.../image_registry`"
configure_storage("azure")       # so every step can read blob, also after a kernel restart

AZ_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/<your-user>/x3-gate4"
assert "<" not in AZ_ROOT, "set your own blob prefix in AZ_ROOT (replace <your-user>)"

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "145_x3_shard_keys"   # local: only the image ref
ROI = CHECKOUT / "notebooks" / "shapefiles" / "s2grid=4772924.geojson"   # inside MGRS tile T33UWP
START = datetime.datetime(2018, 6, 19)   # a 2-day window around one acquisition, 2018-06-20
END = datetime.datetime(2018, 6, 21)
BANDS = ["B04"]                          # one band: one asset per granule
COLLECTION = "sentinel-2-l2a"
ARCHIVE = f"{AZ_ROOT}/imagery"           # its own archive, so nothing is skipped as already present
CATALOG = fsd.archive_catalog_filepath(ARCHIVE, COLLECTION)
RUN_ROOT = f"{AZ_ROOT}/runs-root"        # where the shard CSV and _status/ JSON land

## Step 1 — build (or reuse) an image from this checkout

**What it does:** declares the AML image with `fsd="path:<this checkout>"`, so the node runs this
branch's code, and asks `fsd.aml.ensure_environment` for it. `alias=None` (the default): no shared
registry pointer such as `current` moves (`CONTRIBUTING.md`).

**PASS if:** the environment is reused, i.e. its image is already built. The first run usually
**builds**: the cell then stops with a link. Open it, wait for **Build status: Succeeded**
(~10-20 min), and run the cell again; it then says `reused` and records the image ref on disk.

In [ ]:
# Step 1 -- ensure the environment for THIS checkout's fsd.
import fsd.aml
from fsd.image import ImageDefinition

BASE = ImageDefinition(name="fsd-aml-env", fsd=f"path:{CHECKOUT}", extras=("local", "azure", "mpc"))
env = fsd.aml.ensure_environment(
    BASE, registry=cfg.image_registry,
    resource_group=cfg.resource_group, workspace=cfg.workspace, storage="azure",
)
print(env.ref, "reused" if env.reused else "JUST BUILT")
assert env.reused, (
    f"{env.ref} was just built: open {env.build_url} , wait for `Build status: Succeeded`, "
    "then re-run this cell.")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "env.json").write_text(json.dumps({"ref": env.ref, "registry_ref": env.registry_ref}))

## Step 2 — download one granule through one MPC shard job

**What it does:** `fsd.download(..., runner="aml")`. Discovery runs on this machine and writes
one shard CSV; one AML job runs `python -m fsd.workflows.download --shard ...` with the Step 1
image, which reads the CSV and calls `download_shard` (the code X3 changed).

**PASS if:** the call returns without raising (the driver raises `RuntimeError: job(s)/shard(s)
failed` if the node job fails). Re-running after a success submits nothing: the catalog already
holds the granule, and it prints `0 of N assets missing`.
**Time:** ~5-10 min (estimate), mostly node start.

In [ ]:
# Step 2 -- one MPC shard on AML, with the image recorded in Step 1.
env_ref = json.loads((OUT / "env.json").read_text())["ref"]
catalog_fp = fsd.download(
    roi=str(ROI), startdate=START, enddate=END, bands=BANDS, dst_folderpath=ARCHIVE,
    source="mpc", collection=COLLECTION, max_tiles=2, storage="azure",
    runner="aml",
    runner_kwargs=dict(
        cluster=cfg.cluster, environment=env_ref, root=RUN_ROOT,
        identity_client_id=cfg.uami_client_id,       # the NODES' identity, not your login
        subscription_id=cfg.subscription_id, resource_group_name=cfg.resource_group,
        workspace_name=cfg.workspace, n_shards=1, poll_interval_seconds=10,
    ),
)
print(catalog_fp)

## Step 3 — check the catalog rows the node wrote

**What it does:** reads the archive's catalog back from blob.

**PASS if:** it holds at least one granule, and every row carries the values the driver sent, not
a default: `acquisition_key` is set and differs from the granule name (the old default for a missing
key was the granule name itself); `processing_version` and `processing_datetime` are set; `properties`
holds the real STAC properties (the old default was `"{}"`); `scale` is set; `files` includes `B04.tif`.

In [ ]:
# Step 3 -- read back and assert.
from fsd.catalog.catalog import TileCatalog

gdf = TileCatalog(CATALOG).read()
assert len(gdf) >= 1, f"expected >= 1 catalog row in {CATALOG}, found 0"
for _, r in gdf.iterrows():
    assert r["acquisition_key"] and r["acquisition_key"] != r["id"], (
        f"{r['id']}: acquisition_key {r['acquisition_key']!r} should be set and differ from the granule name")
    assert r["processing_version"], f"{r['id']}: processing_version is empty"
    assert r["processing_datetime"] is not None and str(r["processing_datetime"]) != "NaT", (
        f"{r['id']}: processing_datetime is empty")
    assert r["properties"] not in (None, "", "{}"), f"{r['id']}: properties is empty"
    assert r["scale"] == r["scale"], f"{r['id']}: scale is NaN"
    assert "B04.tif" in r["files"].split(","), f"{r['id']}: files {r['files']!r} lacks B04.tif"
print(f"PASS: {len(gdf)} row(s)")

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart.

In [ ]:
from fsd.catalog.catalog import TileCatalog

gdf = TileCatalog(CATALOG).read()
summary = {
    "image": json.loads((OUT / "env.json").read_text()),
    "catalog": CATALOG,
    "rows": [
        {"id": r["id"], "acquisition_key": r["acquisition_key"],
         "processing_version": r["processing_version"],
         "processing_datetime": str(r["processing_datetime"]), "scale": r["scale"],
         "files": r["files"], "properties_chars": len(r["properties"] or "")}
        for _, r in gdf.iterrows()
    ],
}
print(json.dumps(summary, indent=2, default=str))